# 🚀 LoRA 推理部署 — Multi-LoRA Serving

**本文目标**：掌握 LoRA adapter 在生产推理框架中的部署——vLLM/SGLang 的 multi-LoRA 支持。

读完这篇你会理解：
- vLLM 的 Punica 内核: 如何高效服务多个 LoRA adapter
- SGLang 的 LoRA 支持
- Multi-LoRA 的显存规划
- "热加载" LoRA 的延迟和吞吐 trade-off

## 1. Multi-LoRA 的挑战

### 1.1 问题

```
场景: 一个 SaaS 平台, 每个客户有自己微调的 LoRA adapter
  → 100 个客户 = 100 个 LoRA adapter
  → 每个请求可能用不同的 adapter
  → 不能每个 adapter 部署一个独立服务 (成本太高)

Multi-LoRA Serving: 一个 base model + N 个 LoRA adapter
  → 共享 base model 的 KV Cache 和权重
  → 每个请求动态选择对应的 LoRA adapter
```

### 1.2 技术挑战

```
挑战 1: 显存
  100 个 LoRA adapter × ~10MB (rank=16) = ~1 GB → 还好
  1000 个 LoRA adapter × ~10MB = ~10 GB → 需要注意
  → 总 adapter 显存 = N × rank × d_model × 2 × 2 bytes

挑战 2: 计算
  B@A@x 是一个额外的矩阵乘法
  → 每次 decode step 都要执行
  → 如果 adapter 切换频繁 → GPU kernel launch overhead

挑战 3: Batching
  同一 batch 内的请求可能用不同 LoRA adapter
  → 不能简单地把所有请求的 LoRA 权重合并
  → 需要"分段"计算: 相同 adapter 的请求一起算
```

## 2. vLLM Multi-LoRA — Punica 内核

### 2.1 Punica 的设计

```
Punica (vLLM 的 multi-LoRA 内核):

核心思想: 把 batch 按 adapter 分组, 每组用不同的 LoRA 权重

Batch = [req_A(adapter_1), req_B(adapter_2), req_C(adapter_1), req_D(adapter_3)]

Step 1: 收集相同 adapter 的请求
  group_1: [req_A, req_C] → adapter_1
  group_2: [req_B]        → adapter_2
  group_3: [req_D]        → adapter_3

Step 2: 每组分别计算 B@A@x
  for group in groups:
    lora_out[group] = (x[group] @ A.T) @ B.T  # 每组用不同的 A,B

Step 3: 合并结果
  output = base_out + lora_out

SGMV (Segmented Gather Matrix-Vector) kernel:
  → 在 GPU 上高效实现上述分组计算
  → 用 CUDA 的 warp-level 并行处理不同 adapter
```

### 2.2 配置和启动

```bash
# vLLM Multi-LoRA 配置
vllm serve meta-llama/Llama-3-8B-Instruct     --enable-lora     --max-lora-rank 64     --max-loras 128     --max-cpu-loras 512     --lora-modules customer-1=./lora/llama3-customer1                   customer-2=./lora/llama3-customer2                   coding-assistant=./lora/llama3-code

# 请求时指定 adapter
curl http://localhost:8000/v1/chat/completions   -H "Content-Type: application/json"   -d '{
    "model": "customer-1",
    "messages": [{"role": "user", "content": "Hello"}]
  }'
```

### 2.3 显存规划

```
vLLM Multi-LoRA 的显存布局:

┌──────────────────────────────────────────────────────┐
│  Base Model Weights  │  KV Cache Pool               │
│  (共享, FP16/BF16)   │  (PagedAttention)            │
├──────────────────────┼──────────────────────────────┤
│  LoRA Adapters        │  激活值 + 临时 Buffer       │
│  - adapter_1: ~10MB   │                              │
│  - adapter_2: ~10MB   │                              │
│  - ... (N 个)          │                              │
│  (可通过 CPU swap)     │                              │
└──────────────────────────────────────────────────────┘

--max-loras:           GPU 上缓存的 adapter 数量
--max-cpu-loras:       总 adapter 数量 (不常用在 CPU)
--max-lora-rank:       允许的最大 rank (决定显存预留)

策略:
  常用 adapter → 常驻 GPU
  不常用 → GPU 缓存池 (LRU 淘汰, 从 CPU 换入 ~1ms)
```

## 3. SGLang LoRA 支持

```bash
# SGLang 的 LoRA 支持 (基于 RadixAttention)

python -m sglang.launch_server     --model-path meta-llama/Llama-3-8B-Instruct     --enable-lora     --max-lora-rank 64     --lora-paths customer1=./lora/c1 customer2=./lora/c2

# SGLang 的优势:
# RadixAttention 的 prefix cache 可以跨 LoRA adapter 共享!
# → 如果多个 adapter 有相同的 system prompt
# → KV Cache 不会被 adapter 差异影响
```

## 4. 性能考量

| 指标 | 无 LoRA | 1 Adapter | 10 Adapters | 100 Adapters |
|------|--------|-----------|------------|-------------|
| 延迟 overhead | — | ~2-3% | ~3-5% | ~5-8% |
| 显存 overhead | — | ~10MB/adapter | ~100MB | ~1GB |
| 吞吐影响 | — | ~2% | ~3-5% | ~5-10% |
| adapter 切换延迟 | — | 0 | ~0.1ms | ~1ms (GPU→CPU→GPU) |

> overhead 来源于 SGMV kernel 的额外计算和 adapter 分组开销
> 10 个 adapter 以下 → 几乎无感知
> 100 个 adapter → 需要关注显存和调度

## 5. LoRA 推理的最佳实践

```
1. Merge 部署 (推荐, 如果不需要多 adapter):
   → 合并后推理 = 原始模型的速度 + 0% overhead
   → 适合: 单一模型/单一客户场景

2. Multi-LoRA (适合 SaaS 平台):
   → 10 个以内: GPU 缓存, 几乎无开销
   → 100 个以内: 常用 GPU + 不常用 CPU
   → 1000 个: 需要专门的 adapter 管理服务

3. Rank 的选择 (推理视角):
   → rank=8: 推荐 (推理 overhead < 2%)
   → rank=16: 可接受 (推理 overhead ~3%)
   → rank=64: 慎重 (推理 overhead ~10%, 显存 40MB/adapter)

4. Adapter 的热加载:
   vLLM: 支持运行时动态加载新 adapter (--enable-lora)
   SGLang: 需要重启 (当前版本)
   llama.cpp: 不支持 unmerged LoRA (需要手动 merge)
```